# NB04 - S1-only U-Net Baseline (VV/VH -> flood mask)

Controlled **image-only baseline**: standard U-Net, input = Sentinel-1 VV/VH (2 ch, 512x512), output = 1-ch flood logits. DEM/rainfall/S2 are never passed to the model. Train on train, select on valid IoU, evaluate on test ONCE. Bolivia reserved for NB07 (loader smoke-test only, no metrics for decisions).

Loss = 0.5 * masked-BCE + 0.5 * masked-Dice, all over `valid_mask` pixels only. Interpretation: this notebook *establishes the baseline* - no multimodal comparison is made here.

## 1. Load project context

In [ ]:
from pathlib import Path
CTX = Path(r"C:/Users/Swarnim/Desktop/ML projects/tsao/project_context")
for f in ["NEXT_STEPS.md", "DECISIONS.md"]:
    print("=" * 70)
    print(f"### {f} (head) ###")
    print("\n".join((CTX / f).read_text(encoding="utf-8").splitlines()[:12]))
print("\nD11 reminder:", [ln for ln in (CTX / "DECISIONS.md").read_text(encoding="utf-8").splitlines() if ln.startswith("## D11")])


## 2. Imports + configuration

In [ ]:
from pathlib import Path
import json, random, time, datetime
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
PROJECT_ROOT = Path(r"C:/Users/Swarnim/Desktop/ML projects/tsao")
PROCESSED_ROOT = PROJECT_ROOT / "processed"
MODEL_DIR = PROJECT_ROOT / "models"; MODEL_DIR.mkdir(exist_ok=True)
RESULTS_DIR = PROJECT_ROOT / "results"; RESULTS_DIR.mkdir(exist_ok=True)
BEST_CKPT = MODEL_DIR / "nb04_s1_unet_best.pt"
CONFIG_JSON = MODEL_DIR / "nb04_s1_unet_config.json"
HIST_CSV = RESULTS_DIR / "nb04_training_history.csv"
RESULTS_JSON = RESULTS_DIR / "nb04_results.json"
SEED = 42; BATCH_SIZE = 4; LR = 1e-3; WEIGHT_DECAY = 1e-4; EPOCHS = 40; PATIENCE = 8
BASE_CH = 32; IN_CH = 2  # S1-only: VV + VH, nothing else
LOSS_W = {"bce": 0.5, "dice": 0.5}; THR = 0.5; EPS = 1e-6
USE_AMP = torch.cuda.is_available()
print(f"epochs={EPOCHS} batch={BATCH_SIZE} lr={LR} wd={WEIGHT_DECAY} patience={PATIENCE} base_ch={BASE_CH} amp={USE_AMP}")


## 3. Reproducibility / device setup

In [ ]:
import random
import numpy as np, torch
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True  # speed over bit-determinism on RTX 4050; seeds still fixed (documented)
print(f"device={DEVICE} | torch={torch.__version__} | gpu={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")
g = torch.Generator().manual_seed(SEED)


## 4. Load NB03 Dataset / DataLoaders
Same `FloodDataset` logic as NB03 (verified class, copied verbatim - no preprocessing changes), wrapped by `S1Subset` which slices channels 0:2 after asserting they are VV/VH per `normalization_stats.json`. DEM/rain cannot leak through.

In [ ]:
import json
import numpy as np, torch
from torch.utils.data import Dataset, DataLoader
st = json.loads((PROCESSED_ROOT / "normalization_stats.json").read_text())
assert st["split"] == "train" and st["channels"][:2] == ["VV_dB", "VH_dB"], st["channels"]
print("train stats channels:", st["channels"])

class FloodDataset(Dataset):  # identical to NB03 (locked) - do not alter logic
    def __init__(self, split, processed_root=PROCESSED_ROOT, stats_path=PROCESSED_ROOT / "normalization_stats.json"):
        assert split in ("train", "valid", "test", "bolivia"), split
        s = json.loads(stats_path.read_text())
        assert s["split"] == "train", "normalization stats must be train-derived"
        self.mean = np.array(s["mean"], dtype=np.float32).reshape(-1, 1, 1)
        self.std = np.array(s["std"], dtype=np.float32).reshape(-1, 1, 1)
        assert (self.std > 0).all()
        self.paths = sorted((processed_root / split).glob("*.npz"))
        assert len(self.paths) > 0
        self.split = split
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        d = np.load(self.paths[i], allow_pickle=True)
        x = (d["X"].astype(np.float32) - self.mean) / self.std
        x = np.where(np.isfinite(x), x, 0.0).astype(np.float32)
        return {"x": torch.from_numpy(x), "y": torch.from_numpy(d["y"].astype(np.float32)),
                "mask": torch.from_numpy(d["valid_mask"].astype(bool)), "chip_id": str(d["chip_id"])}

class S1Subset(Dataset):  # NB04 wrapper: VV/VH only, verified by construction
    def __init__(self, base): self.base = base
    def __len__(self): return len(self.base)
    def __getitem__(self, i):
        r = dict(self.base[i])
        assert r["x"].shape[0] == 6, "expected 6-channel NB03 sample"
        r = dict(r); r["x"] = r["x"][:IN_CH]
        assert r["x"].shape == (2, 512, 512)
        return r

base = {s: FloodDataset(s) for s in ("train", "valid", "test", "bolivia")}
ds = {s: S1Subset(base[s]) for s in base}
print({s: len(ds[s]) for s in ds})
assert {s: len(ds[s]) for s in ds} == {"train": 252, "valid": 89, "test": 90, "bolivia": 15}
loaders = {
    "train": DataLoader(ds["train"], batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available(), generator=g),
    "valid": DataLoader(ds["valid"], batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available()),
    "test": DataLoader(ds["test"], batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available()),
    "bolivia": DataLoader(ds["bolivia"], batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available()),
}
mm = pd.read_csv(PROCESSED_ROOT / "metadata.csv")
print(f"train flood% (NB02 valid-px, mean over chips): {mm.loc[mm['split']=='train','flood_pct_valid'].mean():.3f} (imbalance context; no weighting used)")


## 5. S1-only channel selection
Explicit gate: the model-facing batch has exactly 2 channels, statistically VV/VH.

In [ ]:
b = next(iter(loaders["train"]))
assert tuple(b["x"].shape) == (BATCH_SIZE, 2, 512, 512), b["x"].shape
assert st["channels"][:2] == ["VV_dB", "VH_dB"]
print(f"MODEL INPUT VERIFIED: {tuple(b['x'].shape)} - VV/VH only. DEM/rainfall channels are dropped before the model.")
del b


## 6. U-Net architecture
Standard encoder-bottleneck-decoder with skip connections (2→32→64→128→256→[512]→256→128→64→32→1). BatchNorm+ReLU, MaxPool down, ConvTranspose2d up. Logits out (no sigmoid - BCEWithLogits).

In [ ]:
import torch.nn as nn
class DoubleConv(nn.Module):
    def __init__(self, ci, co):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(ci, co, 3, padding=1, bias=False), nn.BatchNorm2d(co), nn.ReLU(inplace=True),
                                 nn.Conv2d(co, co, 3, padding=1, bias=False), nn.BatchNorm2d(co), nn.ReLU(inplace=True))
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_ch=2, base=32):
        super().__init__()
        self.e1 = DoubleConv(in_ch, base);      self.p1 = nn.MaxPool2d(2)
        self.e2 = DoubleConv(base, base * 2);   self.p2 = nn.MaxPool2d(2)
        self.e3 = DoubleConv(base * 2, base * 4); self.p3 = nn.MaxPool2d(2)
        self.e4 = DoubleConv(base * 4, base * 8); self.p4 = nn.MaxPool2d(2)
        self.neck = DoubleConv(base * 8, base * 16)
        self.u4 = nn.ConvTranspose2d(base * 16, base * 8, 2, 2);  self.d4 = DoubleConv(base * 16, base * 8)
        self.u3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2);   self.d3 = DoubleConv(base * 8, base * 4)
        self.u2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2);    self.d2 = DoubleConv(base * 4, base * 2)
        self.u1 = nn.ConvTranspose2d(base * 2, base, 2, 2);        self.d1 = DoubleConv(base * 2, base)
        self.out = nn.Conv2d(base, 1, 1)
    def forward(self, x):
        x1 = self.e1(x); x2 = self.e2(self.p1(x1)); x3 = self.e3(self.p2(x2)); x4 = self.e4(self.p3(x3))
        n = self.neck(self.p4(x4))
        m = self.d4(torch.cat([self.u4(n), x4], 1)); m = self.d3(torch.cat([self.u3(m), x3], 1))
        m = self.d2(torch.cat([self.u2(m), x2], 1)); m = self.d1(torch.cat([self.u1(m), x1], 1))
        return self.out(m)  # logits (B,1,512,512)

model = UNet(IN_CH, BASE_CH).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"UNet-32 params: {n_params:,} | in=(B,2,512,512) out=(B,1,512,512)")
x0 = torch.randn(1, 2, 512, 512, device=DEVICE)
with torch.no_grad(): y0 = model(x0)
assert tuple(y0.shape) == (1, 1, 512, 512)
print("forward OK", tuple(y0.shape))
del x0, y0


## 7. Masked BCE + Dice loss
Over valid pixels V (`mask==True`) with logits L, targets T in {0,1}, P=sigmoid(L):
BCE = mean over V of `BCEWithLogits(L,T)`; Dice = 1 - (2·Σ(V·P·T)+eps)/(Σ(V·P)+Σ(V·T)+eps) (batch-global soft Dice, valid pixels only); Loss = 0.5·BCE + 0.5·Dice. Invalid pixels contribute exactly zero.

In [ ]:
import torch.nn.functional as F
def masked_bce_dice(logits, target, mask, w_bce=0.5, w_dice=0.5, eps=1e-6):
    m = mask.float()
    n = m.sum().clamp_min(1.0)
    bce = (F.binary_cross_entropy_with_logits(logits, target, reduction="none") * m).sum() / n
    p = torch.sigmoid(logits)
    inter = (p * target * m).sum()
    dice = 1.0 - (2.0 * inter + eps) / ((p * m).sum() + (target * m).sum() + eps)
    return w_bce * bce + w_dice * dice, {"bce": bce.detach(), "dice": dice.detach()}


## 8. Metrics
Global (aggregate) TP/FP/FN/TN over valid pixels at thr 0.5; IoU/F1/precision/recall with eps-safe division. Aggregate-from-totals (not mean-of-chips) is the reported metric.

In [ ]:
@torch.no_grad()
def accumulate_metrics(logits, target, mask, acc, thr=0.5, eps=1e-9):
    pred = (torch.sigmoid(logits) >= thr).float()
    t, m = target.float(), mask.float()
    acc["tp"] += float(((pred == 1) & (t == 1) & (m == 1)).sum())
    acc["fp"] += float(((pred == 1) & (t == 0) & (m == 1)).sum())
    acc["fn"] += float(((pred == 0) & (t == 1) & (m == 1)).sum())
    acc["tn"] += float(((pred == 0) & (t == 0) & (m == 1)).sum())

def summarize(acc, eps=1e-9):
    tp, fp, fn, tn = (acc[k] for k in ("tp", "fp", "fn", "tn"))
    iou = tp / (tp + fp + fn + eps)
    prec = tp / (tp + fp + eps); rec = tp / (tp + fn + eps)
    f1 = 2 * prec * rec / (prec + rec + eps)
    return {"iou": iou, "f1": f1, "precision": prec, "recall": rec,
            "tp": tp, "fp": fp, "fn": fn, "tn": tn}


## 9. Sanity checks
Batch → forward → loss → backward → optimizer step, all finite; plus the masked-loss invariance test.

In [ ]:
model.train()
opt0 = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
b = next(iter(loaders["train"]))
xb, yb, mb = b["x"].to(DEVICE), b["y"].unsqueeze(1).to(DEVICE), b["mask"].unsqueeze(1).to(DEVICE)
assert xb.shape[1] == 2, "model must see exactly 2 channels"
opt0.zero_grad()
out = model(xb)
assert tuple(out.shape) == (BATCH_SIZE, 1, 512, 512)
loss, parts = masked_bce_dice(out, yb, mb, LOSS_W["bce"], LOSS_W["dice"])
loss.backward()
assert torch.isfinite(loss) and torch.isfinite(out).all()
assert all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)
opt0.step()
print(f"sanity: fwd {tuple(out.shape)} loss={float(loss):.4f} (bce={float(parts['bce']):.4f} dice={float(parts['dice']):.4f}) bwd+step OK, all finite")
# masked-loss invariance: perturbing ONLY invalid pixels must not change the loss
torch.manual_seed(0)
Lt, Tt = torch.randn(2, 1, 32, 32), (torch.rand(2, 1, 32, 32) > 0.9).float()
Mt = torch.ones(2, 1, 32, 32); Mt[:, :, :8, :8] = 0  # 25% invalid block
l1, _ = masked_bce_dice(Lt, Tt, Mt)
Lt2, Tt2 = Lt.clone(), Tt.clone()
Lt2[~Mt.bool()] = 999.0; Tt2[~Mt.bool()] = 1.0 - Tt2[~Mt.bool()]
l2, _ = masked_bce_dice(Lt2, Tt2, Mt)
assert l1.item() == l2.item(), (l1.item(), l2.item())
print(f"masked-loss invariance: loss identical ({float(l1):.6f}) after corrupting invalid pixels only - PASS")
del b, xb, yb, mb, out, loss


## 10. Tiny overfit test
Fresh model, 1 train batch, 40 AdamW steps - loss must collapse. Gates full training.

In [ ]:
torch.manual_seed(SEED)
probe = UNet(IN_CH, BASE_CH).to(DEVICE).train()
popt = torch.optim.AdamW(probe.parameters(), lr=1e-3, weight_decay=0.0)
bt = next(iter(loaders["train"]))
px, py, pm = bt["x"][:2].to(DEVICE), bt["y"][:2].unsqueeze(1).to(DEVICE), bt["mask"][:2].unsqueeze(1).to(DEVICE)
hist = []
for s in range(60):
    popt.zero_grad()
    l, _ = masked_bce_dice(probe(px), py, pm)
    l.backward(); popt.step()
    hist.append(float(l))
print(f"overfit: {hist[0]:.4f} -> {hist[-1]:.4f} (x{hist[0] / max(hist[-1], 1e-9):.1f} reduction)")
assert hist[-1] < 0.6 * hist[0], "model cannot overfit 1 batch - STOP"
print("OVERFIT GATE PASS - capacity/gradients healthy")
del probe, popt, bt, px, py, pm
torch.manual_seed(SEED)  # re-seed so full training starts from the canonical init
model = UNet(IN_CH, BASE_CH).to(DEVICE)


## 11. Training loop
AdamW + ReduceLROnPlateau (val loss) + AMP + early stopping on val IoU (patience 8). Test never touched here.

In [ ]:
import time
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=4)
scaler = torch.amp.GradScaler("cuda") if USE_AMP else None
history, best_iou, bad, t_start = [], -1.0, 0, time.time()

@torch.no_grad()
def evaluate(split):
    model.eval()
    acc = {"tp": 0.0, "fp": 0.0, "fn": 0.0, "tn": 0.0}
    tot, n = 0.0, 0.0
    for b in loaders[split]:
        x = b["x"].to(DEVICE, non_blocking=True)
        y = b["y"].unsqueeze(1).to(DEVICE, non_blocking=True)
        m = b["mask"].unsqueeze(1).to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            out = model(x)
            l, _ = masked_bce_dice(out, y, m)
        tot += float(l) * x.shape[0]; n += x.shape[0]
        accumulate_metrics(out.float(), y, m, acc, THR)
    return {"loss": tot / max(n, 1), **summarize(acc)}

for ep in range(1, EPOCHS + 1):
    model.train()
    tl, n = 0.0, 0
    for b in loaders["train"]:
        x = b["x"].to(DEVICE, non_blocking=True)
        y = b["y"].unsqueeze(1).to(DEVICE, non_blocking=True)
        m = b["mask"].unsqueeze(1).to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            out = model(x)
            l, _ = masked_bce_dice(out, y, m, LOSS_W["bce"], LOSS_W["dice"])
        assert torch.isfinite(l), f"non-finite train loss at epoch {ep}"
        if scaler: scaler.scale(l).backward(); scaler.step(opt); scaler.update()
        else: l.backward(); opt.step()
        tl += float(l) * x.shape[0]; n += x.shape[0]
    vm = evaluate("valid")
    lr_now = opt.param_groups[0]["lr"]
    sched.step(vm["loss"])
    history.append({"epoch": ep, "train_loss": tl / n, "val_loss": vm["loss"], "val_iou": vm["iou"],
                    "val_f1": vm["f1"], "val_precision": vm["precision"], "val_recall": vm["recall"], "lr": lr_now})
    improved = vm["iou"] > best_iou
    if improved:
        best_iou, bad = vm["iou"], 0
        torch.save({"state_dict": model.state_dict(), "epoch": ep, "val": vm}, BEST_CKPT)
    else:
        bad += 1
    print(f"ep {ep:02d} train={tl/n:.4f} val_loss={vm['loss']:.4f} iou={vm['iou']:.4f} f1={vm['f1']:.4f} "
          f"p={vm['precision']:.3f} r={vm['recall']:.3f} lr={lr_now:.1e} {'*BEST*' if improved else f'(bad {bad})'}", flush=True)
    if bad >= PATIENCE:
        print(f"early stopping at epoch {ep} (patience {PATIENCE})")
        break
train_time_s = time.time() - t_start
pd.DataFrame(history).to_csv(HIST_CSV, index=False)
print(f"\nTRAIN DONE: {len(history)} epochs, {train_time_s/60:.1f} min, best val IoU={best_iou:.4f} -> {HIST_CSV}")


## 12. Validation
Best-checkpoint recap (selection used valid IoU only - test still unseen).

In [ ]:
ckpt = torch.load(BEST_CKPT, map_location=DEVICE, weights_only=False)
print(f"best epoch: {ckpt['epoch']} | val: " + ", ".join(f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}" for k, v in ckpt["val"].items()))
print(f"epochs run: {len(history)} / {EPOCHS} | early-stopped: {len(history) < EPOCHS}")


## 13. Best checkpoint selection
Already saved by val-IoU during training (single checkpoint, no test influence). Config snapshot below.

In [ ]:
cfg = {"model": "UNet", "in_channels": IN_CH, "base_channels": BASE_CH,
       "arch": "2-32-64-128-256-[512]-256-128-64-32-1, BN+ReLU, MaxPool, ConvTranspose2d, logits out",
       "loss": "0.5*maskedBCE + 0.5*maskedDice (valid_mask only)", "threshold": THR,
       "optimizer": "AdamW", "lr": LR, "weight_decay": WEIGHT_DECAY, "scheduler": "ReduceLROnPlateau(min,val_loss,f=0.5,p=4)",
       "batch_size": BATCH_SIZE, "epochs_run": len(history), "epochs_max": EPOCHS, "early_stopping_patience": PATIENCE,
       "selection": "best validation IoU", "best_epoch": ckpt["epoch"], "best_val": ckpt["val"],
       "seed": SEED, "amp": USE_AMP, "device": str(DEVICE),
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
       "torch": torch.__version__, "cuda_available": torch.cuda.is_available(),
       "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat()}
CONFIG_JSON.write_text(json.dumps(cfg, indent=2))
print(json.dumps({k: cfg[k] for k in ("best_epoch", "best_val", "epochs_run")}, indent=2))


## 14. Training curves
Loss / IoU / F1 / LR vs epoch (history CSV is the artifact; plots are inspection).

In [ ]:
import matplotlib.pyplot as plt
h = pd.DataFrame(history)
fig, ax = plt.subplots(2, 2, figsize=(12, 7))
ax[0, 0].plot(h["epoch"], h["train_loss"], label="train"); ax[0, 0].plot(h["epoch"], h["val_loss"], label="valid")
ax[0, 0].set_title("loss (masked BCE+Dice)"); ax[0, 0].legend()
ax[0, 1].plot(h["epoch"], h["val_iou"], label="IoU"); ax[0, 1].plot(h["epoch"], h["val_f1"], label="F1")
ax[0, 1].set_title("validation IoU/F1"); ax[0, 1].legend()
ax[1, 0].plot(h["epoch"], h["val_precision"], label="prec"); ax[1, 0].plot(h["epoch"], h["val_recall"], label="rec")
ax[1, 0].set_title("validation precision/recall"); ax[1, 0].legend()
ax[1, 1].plot(h["epoch"], h["lr"]); ax[1, 1].set_title("learning rate"); ax[1, 1].set_yscale("log")
for a in ax.ravel(): a.set_xlabel("epoch")
fig.suptitle(f"NB04 S1-UNet training (best val IoU={best_iou:.4f} @ep {ckpt['epoch']})")
fig.tight_layout(); plt.show()


## 15. Final test evaluation
Load BEST validation checkpoint, evaluate on TEST exactly once. Test played no role in any decision above.

In [ ]:
model.load_state_dict(torch.load(BEST_CKPT, map_location=DEVICE, weights_only=False)["state_dict"])
test_m = evaluate("test")
print("TEST (best-val checkpoint, single evaluation):")
print(", ".join(f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v:.0f}" for k, v in test_m.items()))
assert len(loaders["test"].dataset) == 90  # test split intact, evaluated once
# Bolivia: loader smoke-test only (iteration works) - NO metrics, NO decisions (reserved for NB07)
bb = next(iter(loaders["bolivia"]))
assert tuple(bb["x"].shape[1:]) == (2, 512, 512)
print(f"Bolivia loader smoke OK {tuple(bb['x'].shape)} - no metrics computed (NB07 territory)")
del bb


## 16. Qualitative predictions
Fixed chips: 3 valid + 2 test (first sorted files - deterministic). VV/VH (physical dB from raw npz for readability) | GT (grey=invalid) | predicted probability | binary mask | error map (green=TP, red=FP, blue=FN, black=TN, grey=invalid).

Note: 6 chips have `valid_frac == 0` (5 Ghana chips whose label is entirely -1 = no ground truth; Paraguay_34417 test chip whose S1 is 100% NaN despite a 100%-flood label). They contribute zero pixels to loss/metrics by construction and are skipped for display only (metrics still aggregate over all chips).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
model.eval()
mp = pd.read_csv(PROCESSED_ROOT / "metadata.csv")  # display only: skip zero-valid chips (see note)
def _shown(split, idxs):
    ok = mp[(mp["split"] == split) & (mp["valid_frac"] > 0)].sort_values("chip_id")["chip_id"].tolist()
    return [(split, ok[i]) for i in idxs]
shown = _shown("valid", (0, 89 // 3, 2 * 89 // 3)) + _shown("test", (0, 89 // 2))  # deterministic, event-spread
err_cmap = ListedColormap(["#000000", "#00c000", "#ff0000", "#0060ff", "#808080"])  # TN TP FP FN invalid
for split, stem in shown:
    raw = np.load(PROCESSED_ROOT / split / (stem + ".npz"), allow_pickle=True)
    xb = torch.from_numpy(((raw["X"][:2].astype(np.float32) - np.array(st["mean"][:2], dtype=np.float32).reshape(-1,1,1)) / np.array(st["std"][:2], dtype=np.float32).reshape(-1,1,1))).unsqueeze(0).to(DEVICE)
    with torch.no_grad(), torch.amp.autocast("cuda", enabled=USE_AMP):
        prob = torch.sigmoid(model(xb))[0, 0].cpu().numpy()
    y = raw["y"].astype(np.float32); m = raw["valid_mask"].astype(bool)
    pred = (prob >= THR).astype(np.uint8)
    err = np.full_like(y, 0, dtype=np.uint8); err[(pred==1)&(y==1)&m]=1; err[(pred==1)&(y==0)&m]=2; err[(pred==0)&(y==1)&m]=3; err[~m]=4
    fig, ax = plt.subplots(1, 6, figsize=(17, 3.2))
    ax[0].imshow(raw["X"][0], cmap="gray"); ax[0].set_title(f"VV dB {stem}")
    ax[1].imshow(raw["X"][1], cmap="gray"); ax[1].set_title("VH dB")
    ax[2].imshow(np.where(m, y, -1), cmap=ListedColormap(["#808080","#1a1a1a","#00b3ff"]), vmin=-1, vmax=1, interpolation="nearest")
    ax[2].set_title(f"GT flood={(y[m]==1).mean()*100:.1f}%")
    ax[3].imshow(prob, cmap="Blues", vmin=0, vmax=1); ax[3].set_title("P(flood)")
    ax[4].imshow(pred, cmap="gray", vmin=0, vmax=1, interpolation="nearest"); ax[4].set_title("pred@0.5")
    ax[5].imshow(err, cmap=err_cmap, vmin=0, vmax=4, interpolation="nearest"); ax[5].set_title("TP green/FP red/FN blue")
    for a in ax: a.set_xticks([]); a.set_yticks([])
    fig.suptitle(f"{split}: {stem}")
    fig.tight_layout(); plt.show()
    del xb, prob, y, m, pred, err
print("Qualitative panels above - invalid shown grey, never implied as background.")


## 17. Final results
`results/nb04_results.json` - the NB05/NB06 comparison baseline.

In [ ]:
results = {"notebook": "NB04_S1_UNet_Baseline", "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
           "input": "VV+VH only (2,512,512)", "model": "UNet-32", "params": n_params,
           "loss": "0.5*maskedBCE+0.5*maskedDice", "config": {"seed": SEED, "batch": BATCH_SIZE, "lr": LR, "wd": WEIGHT_DECAY, "epochs_max": EPOCHS, "patience": PATIENCE},
           "train_time_min": round(train_time_s / 60, 2), "epochs_run": len(history), "best_epoch": ckpt["epoch"],
           "best_val": ckpt["val"], "test": test_m, "threshold": THR, "device": str(DEVICE),
           "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "amp": USE_AMP,
           "interpretation": "S1-only baseline for NB05/NB06 comparison; no superiority claim over multimodal learning."}
RESULTS_JSON.write_text(json.dumps(results, indent=2, default=float))
print(json.dumps({"best_val_iou": round(ckpt['val']['iou'], 4), "test_iou": round(test_m['iou'], 4), "test_f1": round(test_m['f1'], 4),
                  "epochs": len(history), "minutes": round(train_time_s/60, 1)}, indent=2))


## 18. Project context update

In [ ]:
import datetime, re
now = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
entry = (f"\n## {now} - NB04_S1_UNet_Baseline COMPLETE\n"
         f"- UNet-32 ({n_params:,} params) VV/VH->logits; masked 0.5BCE+0.5Dice; AdamW lr={LR} wd={WEIGHT_DECAY}; batch={BATCH_SIZE}; AMP={USE_AMP} on {results['gpu']}\n"
         f"- {len(history)} epochs, {results['train_time_min']} min; best ep {ckpt['epoch']} val IoU={ckpt['val']['iou']:.4f} F1={ckpt['val']['f1']:.4f} P={ckpt['val']['precision']:.4f} R={ckpt['val']['recall']:.4f}\n"
         f"- TEST (once): IoU={test_m['iou']:.4f} F1={test_m['f1']:.4f} P={test_m['precision']:.4f} R={test_m['recall']:.4f}\n"
         f"- ckpt models/nb04_s1_unet_best.pt + config; history results/nb04_training_history.csv; results/nb04_results.json\n"
         f"- Masked-loss invariance + overfit gates PASS. Bolivia untouched (loader smoke only). NB04 safe to lock; next NB05.\n")
(CTX / "CHANGELOG.md").write_text((CTX / "CHANGELOG.md").read_text(encoding="utf-8") + entry, encoding="utf-8")
ps = CTX / "PROJECT_STATE.md"
t = ps.read_text(encoding="utf-8")
t = re.sub(r"^\*\*Last updated:\*\*.*$", f"**Last updated:** {now} (NB04 complete: yes)", t, count=1, flags=re.M)
t += f"\n\n## NB04 result ({now})\n- S1-only UNet baseline: val IoU={ckpt['val']['iou']:.4f}, test IoU={test_m['iou']:.4f}; {len(history)} epochs.\n"
ps.write_text(t, encoding="utf-8")
dc = CTX / "DECISIONS.md"
t = dc.read_text(encoding="utf-8")
t = re.sub(r"^\*\*Last updated:\*\*.*$", f"**Last updated:** {now} (D12 baseline)", t, count=1, flags=re.M)
t += "\n## D12 - NB04 image-only baseline (" + now + ")\nS1-only UNet-32, masked BCE+Dice, val-IoU selection, test-once; baseline for NB05/NB06, no superiority claims.\n"
dc.write_text(t, encoding="utf-8")
nx = CTX / "NEXT_STEPS.md"
t = nx.read_text(encoding="utf-8")
t = re.sub(r"^\*\*Last updated:\*\*.*$", f"**Last updated:** {now}", t, count=1, flags=re.M)
t += f"\n\n## Update ({now})\n- NB04 COMPLETE + validated. Next: NB05 multimodal model vs this baseline.\n"
nx.write_text(t, encoding="utf-8")
print(entry)
print("Context files updated:", now)
